# Evaluation Script: Success Rate

This script produces results for Section 5.1 Success Rate

### Connect to the MongoDB

In [ ]:
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")

db = client["webview"]

bytecode_info_collection = db["bytecode_info"]
manifest_info_collection = db["manifest_info"]
network_info_collection = db["network_info"]

TOTAL_APPS_DOWNLOADED = 189779

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

In [2]:
cleartext_package_names = network_info_collection.distinct("package_name", {
    "allowed_domains": {"$in": ["*"]},
    "disallowed_domains": {"$size": 0}
})

webview_package_names = bytecode_info_collection.distinct("package_name", {
    "webViewRelatedMethods.0": {"$exists": True}
})

to_analyze_dynamically = set(cleartext_package_names).intersection(set(webview_package_names))

print_latex_macro("appsToDynamicallyAnalyze", f"{len(to_analyze_dynamically):,}")

\newcommand{\appsToDynamicallyAnalyze}{62,355}


In [3]:
# Static Analysis Success
manifest_analysis_package_names = manifest_info_collection.distinct("package_name")
print_latex_macro("appsManifestSuccess", f"{len(manifest_analysis_package_names):,}")
print_latex_macro("appsManifestSuccessPercentage", f"{len(manifest_analysis_package_names) / TOTAL_APPS_DOWNLOADED * 100:.2f}")

bytecode_analysis_success = bytecode_info_collection.distinct("package_name")
print_latex_macro("appsBytecodeSuccess", f"{len(bytecode_analysis_success):,}")
print_latex_macro("appsBytecodeSuccessPercentage", f"{len(bytecode_analysis_success) / TOTAL_APPS_DOWNLOADED * 100:.2f}")

successful_static_analysis_package_names = set(manifest_analysis_package_names).intersection(set(bytecode_analysis_success))
print_latex_macro("staticAnalysisSuccessfulApps", f"{len(successful_static_analysis_package_names):,}")
print_latex_macro("staticAnalysisSuccessfulAppsPercent", f"{len(successful_static_analysis_package_names)/TOTAL_APPS_DOWNLOADED * 100:.2f}")

static_analysis_success = len(successful_static_analysis_package_names)

\newcommand{\appsManifestSuccess}{189,280}
\newcommand{\appsManifestSuccessPercentage}{99.74}
\newcommand{\appsBytecodeSuccess}{188,069}
\newcommand{\appsBytecodeSuccessPercentage}{99.10}
\newcommand{\staticAnalysisSuccessfulApps}{187,979}
\newcommand{\staticAnalysisSuccessfulAppsPercent}{99.05}


In [4]:
allow_cleartext_traffic_and_webview = len(to_analyze_dynamically)


db0 = pymongo.MongoClient("mongodb://localhost:27017/")["0-wvmc"]
db1 = pymongo.MongoClient("mongodb://localhost:27017/")["1-wvmc"]
db2 = pymongo.MongoClient("mongodb://localhost:27017/")["2-wvmc"]
db3 = pymongo.MongoClient("mongodb://localhost:27017/")["3-wvmc"]

# Check how many we attempted
db0_package_names = set(db0["analysis_status"].distinct("package_name"))
db1_package_names = set(db1["analysis_status"].distinct("package_name"))
db2_package_names = set(db2["analysis_status"].distinct("package_name"))
db3_package_names = set(db3["analysis_status"].distinct("package_name"))
dynamic_attempted_package_names = db0_package_names.union(db1_package_names).union(db2_package_names).union(db3_package_names)
# filter None
# Remember to exclude the "apps"
dynamic_attempted_package_names = [x for x in dynamic_attempted_package_names if x is not None]

# Check how many succeeded
db0_package_names_succeeded = set(db0["analysis_status"].distinct("package_name", {"dynamic.joblog": {"$in": ["0","124"]}, "dynamic.added": True}))
db1_package_names_succeeded = set(db1["analysis_status"].distinct("package_name", {"dynamic.joblog": {"$in": ["0","124"]}, "dynamic.added": True}))
db2_package_names_succeeded = set(db2["analysis_status"].distinct("package_name", {"dynamic.joblog": {"$in": ["0","124"]}, "dynamic.added": True}))
db3_package_names_succeeded = set(db3["analysis_status"].distinct("package_name", {"dynamic.joblog": {"$in": ["0","124"]}, "dynamic.added": True}))
dynamic_package_names_success = db0_package_names_succeeded.union(db1_package_names_succeeded).union(db2_package_names_succeeded).union(db3_package_names_succeeded)

dynamic_analysis_attempted = len(dynamic_attempted_package_names)
print_latex_macro("dynamicAnalysisAttemptedApps", f"{len(dynamic_attempted_package_names):,}")
print_latex_macro("dynamicAnalysisAttemptedAppsPercent", f"{dynamic_analysis_attempted/allow_cleartext_traffic_and_webview*100:.2f}")


dynamic_analysis_successful = len(dynamic_package_names_success)
print_latex_macro("dynamicAnalysisSuccessfulApps", f"{dynamic_analysis_successful:,}")
print_latex_macro("dynamicAnalysisSuccessfulAppsPercent", f"{dynamic_analysis_successful/dynamic_analysis_attempted*100:.2f}")


dynamic_analysis_failed = dynamic_analysis_attempted - dynamic_analysis_successful
dynamic_skipped = allow_cleartext_traffic_and_webview - dynamic_analysis_attempted


\newcommand{\dynamicAnalysisAttemptedApps}{35,000}
\newcommand{\dynamicAnalysisAttemptedAppsPercent}{56.13}
\newcommand{\dynamicAnalysisSuccessfulApps}{34,200}
\newcommand{\dynamicAnalysisSuccessfulAppsPercent}{97.71}
